In [ ]:
# !  = הרץ פקודה בטרמינל (לא Python)
# pip install = כלי התקנת חבילות Python
# -q = quiet, הסתר פלט מיותר
!pip install -q langgraph langchain-anthropic langchain-community tavily-python

In [ ]:
import os  # ספריית Python מובנית לעבודה עם משתני סביבה

try:
    # google.colab.userdata = מודול מיוחד של Colab לגישה ל-Secrets
    from google.colab import userdata

    # os.environ = מילון של משתני סביבה — דרך תקנית להעביר הגדרות לתוכנות
    # המפתח נשמר בזיכרון ולא בקוד עצמו
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    os.environ["TAVILY_API_KEY"]    = userdata.get("TAVILY_API_KEY")

except Exception:
    # אם לא רצים ב-Colab (למשל, מחשב מקומי) — הגדר ידנית:
    # os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
    # os.environ["TAVILY_API_KEY"]    = "tvly-..."
    pass

# os.environ.get() מחזיר את הערך אם קיים, או None אם לא
# הביטוי "✓" if ... else "✗ חסר" = בדיקה קצרה (ternary)
print("ANTHROPIC_API_KEY:", "✓" if os.environ.get("ANTHROPIC_API_KEY") else "✗ חסר")
print("TAVILY_API_KEY:   ", "✓" if os.environ.get("TAVILY_API_KEY")    else "✗ חסר")

In [ ]:
from langchain_anthropic import ChatAnthropic          # מודל Claude
from langchain_community.tools.tavily_search import TavilySearchResults  # כלי חיפוש

# יצירת כלי חיפוש:
# max_results=3 = לכל שאלה, החזר עד 3 דפי אינטרנט רלוונטיים
search_tool = TavilySearchResults(max_results=3)

# רשימת הכלים שיהיו זמינים לסוכן
# שמים ברשימה כי הסוכן יכול לקבל מספר כלים שונים
tools = [search_tool]

# יצירת מודל Claude:
# ChatAnthropic() = מחבר ל-API של Anthropic עם המפתח מ-os.environ
# .bind_tools(tools) = מחבר את הכלים למודל, כך שהוא יכול להפעיל אותם
llm = ChatAnthropic(model="claude-sonnet-4-6").bind_tools(tools)

print("כלי חיפוש:", search_tool.name)   # מדפיס את שם הכלי
print("מודל:     ", "claude-sonnet-4-6") # מאשר איזה מודל נטען

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
# StateGraph  = מחלקה לבניית גרף עם מצב
# MessagesState = סוג מצב מובנה שמנהל רשימת הודעות
# START, END  = קבועים מיוחדים שמסמנים כניסה ויציאה מהגרף

from langgraph.prebuilt import ToolNode
# ToolNode = צומת מוכן שיודע להריץ כלים אוטומטית לפי בקשת המודל


# --- הגדרת הצמתים ---

def llm_node(state: MessagesState):
    # state["messages"] = כל ההיסטוריה של השיחה עד עכשיו
    # llm.invoke() = שולח את ההיסטוריה למודל ומקבל תשובה
    response = llm.invoke(state["messages"])
    # מחזירים מילון שמוסיף את התשובה לרשימת ההודעות
    return {"messages": [response]}

# ToolNode מקבל את רשימת הכלים ויודע לזהות ולהריץ את הכלי הנכון
tool_node = ToolNode(tools)


# --- קשת מותנית ---

def should_continue(state: MessagesState):
    # state["messages"][-1] = ההודעה האחרונה (התשובה הכי חדשה של המודל)
    last = state["messages"][-1]
    # tool_calls = רשימה של כלים שהמודל ביקש להפעיל
    # אם יש בקשות כלים → עבור לצומת "tools"
    # אם אין → סיים את הריצה (END)
    return "tools" if last.tool_calls else END


# --- בניית הגרף ---

# יצירת גרף חדש שמשתמש ב-MessagesState כמצב שלו
graph = StateGraph(MessagesState)

# הוספת הצמתים לגרף (שם + פונקציה)
graph.add_node("llm",   llm_node)
graph.add_node("tools", tool_node)

# קשת קבועה: START תמיד עובר ל-"llm"
graph.add_edge(START, "llm")

# קשת מותנית: מ-"llm" — הפונקציה should_continue מחליטה לאן לעבור
graph.add_conditional_edges("llm", should_continue)

# קשת קבועה: אחרי הרצת כלים — חזור תמיד ל-"llm"
graph.add_edge("tools", "llm")

# compile() = "הקפא" את הגרף לאובייקט מוכן לשימוש
agent = graph.compile()
print("הגרף נבנה בהצלחה ✓")

# הצגה ויזואלית של הגרף (אם Graphviz מותקן)
try:
    from IPython.display import Image, display
    # draw_mermaid_png() = מייצר תמונה של הגרף בפורמט PNG
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    # אם לא עובד — הדפס תיאור טקסטואלי
    print("START → llm → (tool_calls?) → tools → llm → ... → END")

In [ ]:
from langchain_core.messages import HumanMessage
# HumanMessage = סוג הודעה שמייצגת מה המשתמש כתב
# (קיימים גם AIMessage, ToolMessage וכו')

# השאלה שנרצה לשאול את הסוכן
question = "מה הם החידושים הכי מרגשים בתחום ה-AI בשנת 2025?"
print(f"שאלה: {question}\n{'='*60}")

# agent.invoke() = מריץ את הגרף המלא מ-START ועד END
# {"messages": [...]} = מצב ההתחלה — רשימה עם הודעת המשתמש
result = agent.invoke({"messages": [HumanMessage(content=question)]})

# result["messages"] = רשימת כל ההודעות שנוצרו במהלך הריצה
# נעבור עליהן ונדפיס כל שלב בבהירות
for msg in result["messages"]:

    # type(msg).__name__ = שם המחלקה של ההודעה (AIMessage, ToolMessage...)
    # .replace("Message", "") = קיצור לקריאות טובה יותר
    role = type(msg).__name__.replace("Message", "")

    if hasattr(msg, "tool_calls") and msg.tool_calls:
        # המודל ביקש להפעיל כלי — נציג איזה חיפוש ביקש
        for tc in msg.tool_calls:
            # tc["args"]["query"] = מחרוזת החיפוש שהמודל בחר
            print(f"[{role}] 🔍 מחפש: {tc['args'].get('query', '')}")

    elif role == "Tool":
        # הודעת כלי = תוצאות החיפוש שחזרו מ-Tavily
        # (לא מדפיסים את כל הטקסט הגולמי — הוא ארוך)
        print(f"[{role}] תוצאות חיפוש התקבלו")

    else:
        # הודעת AI רגילה — שאלה או תשובה סופית
        print(f"[{role}]\n{msg.content}")

    print("-" * 40)  # קו הפרדה לקריאות